In [1]:
import numpy as np

In [2]:
class State:
    def __init__(self, population, fitness=None):
        self.population = population  # shape: (N, D)
        self.fitness = fitness        # shape: (N,)

    def copy(self):
        return State(
            population=self.population.copy(),
            fitness=None if self.fitness is None else self.fitness.copy()
        )

In [3]:
import sys, time
from colorama import Fore, Style

class ProgressBar:
    def __init__(self, total, length=30):
        self.total = total
        self.length = length

    def update(self, current):
        percent = current / self.total * 100
        bar_len = int(self.length * current / self.total)
        bar = Fore.GREEN+Style.DIM+'—' * bar_len + '>' + Fore.RESET+Style.RESET_ALL
        space = ' ' * (self.length - bar_len)
        print(f"|{bar}{space}| {percent:5.1f}%", end='\r', flush=True)

    def finish(self):
        print()

In [4]:
class Pipeline:
    def __init__(self, operators=None):
        self.operators = operators or []
        self._repeat = 1

    def __rshift__(self, other):
        if isinstance(other, Pipeline):
            return Pipeline(self.operators + other.operators)
        return Pipeline(self.operators + [other])

    def repeat(self, n):
        self._repeat = n
        return self

    def run(self, state=None, verbose=True):
        for i in range(self._repeat):
            if verbose:
                pb = ProgressBar(total=len(self.operators))
                print(f"\nIteration {i+1}/{self._repeat}:")
            for j, op in enumerate(self.operators):
                state = op.apply(state)
                if verbose:
                    pb.update(j + 1)
            if verbose:
                pb.finish()
                if state.fitness is not None:
                    print(f"  Best fitness: {state.fitness.min()}")
        return state

In [5]:
class Operator:
    def apply(self, state):
        raise NotImplementedError

    def __rshift__(self, other):
        return Pipeline([self]) >> other

In [6]:
class Initialize(Operator):
    def __init__(self, fields, count):
        self.fields = fields
        self.count = count

    def apply(self, state=None):
        D = len(self.fields)
        population = np.zeros((self.count, D))

        for d, field in enumerate(self.fields):
            population[:, d] = field.generate(self.count)

        return State(population)

class Initializer:
    def generate(self, size):
        raise NotImplementedError


class Bound(Initializer):
    def __init__(self, low, high):
        self.low = low
        self.high = high

    def generate(self, size):
        return np.random.uniform(self.low, self.high, size)


class Int(Initializer):
    def __init__(self, low, high):
        self.low = low
        self.high = high

    def generate(self, size):
        return np.random.randint(self.low, self.high, size)


class Binary(Initializer):
    def generate(self, size):
        return np.random.choice([0, 1], size)

In [7]:
class Evaluate(Operator):
    def __init__(self, objective_function):
        self.objective_function = objective_function

    def apply(self, state):
        try:
            # try batch
            state.fitness = self.objective_function(state.population)
        except:
            # fallback
            state.fitness = np.array([
                self.objective_function(ind)
                for ind in state.population
            ])
        return state

In [8]:
class GaussianMutation(Operator):
    def __init__(self, sigma=0.1):
        self.sigma = sigma

    def apply(self, state):
        noise = np.random.normal(0, self.sigma, state.population.shape)
        state.population += noise
        return state

In [9]:
class SelectBest(Operator):
    def __init__(self, k, goal='min'):
        self.k = k
        self.goal = goal

    def apply(self, state):
        idx = np.argsort(state.fitness)
        if self.goal == 'max':
            idx = idx[::-1]

        idx = idx[:self.k]

        state.population = state.population[idx]
        state.fitness = state.fitness[idx]
        return state

In [10]:
def sphere(x):
    return np.sum(x**2, axis=1)


fields = [
    Bound(-5, 5),
    Bound(-5, 5),
    Bound(-5, 5),
]

algo = (
    Initialize(fields, count=50)
    >> Evaluate(sphere)
    >> SelectBest(25)
    >> GaussianMutation(0.5)
).repeat(50)

final_state = algo.run()


Iteration 1/50:
|——————————————————————————————>| 100.0%
  Best fitness: 2.806442046806472

Iteration 2/50:
|——————————————————————————————>| 100.0%
  Best fitness: 6.044524780750438

Iteration 3/50:
|——————————————————————————————>| 100.0%
  Best fitness: 2.0881171041647235

Iteration 4/50:
|——————————————————————————————>| 100.0%
  Best fitness: 2.2549423581526735

Iteration 5/50:
|——————————————————————————————>| 100.0%
  Best fitness: 2.174423212771451

Iteration 6/50:
|——————————————————————————————>| 100.0%
  Best fitness: 1.9762812863528834

Iteration 7/50:
|——————————————————————————————>| 100.0%
  Best fitness: 1.8095808182304052

Iteration 8/50:
|——————————————————————————————>| 100.0%
  Best fitness: 3.934795062777736

Iteration 9/50:
|——————————————————————————————>| 100.0%
  Best fitness: 0.17933293324239136

Iteration 10/50:
|——————————————————————————————>| 100.0%
  Best fitness: 3.0727138321892236

Iteration 11/50:
|——————————————————————————————>| 100.0%
  Best fitnes